In [ ]:
import requests
import json
from google.colab import userdata

# API Test Portfolio
api_key = "YOUR_API_KEY_HERE"
portfolio_id = "YOUR_PORTFOLIO_ID_HERE"

url = f"https://api.securityscorecard.io/portfolios/{portfolio_id}/companies"

headers = {
    "accept": "application/json; charset=utf-8",
    "Authorization": f"Token {api_key}"
}

response = requests.get(url, headers=headers)

json_data = response.json()

entries = json_data["entries"]
print("{:<25} {:<25} {:<10} {:<5}".format("Name", "Domain", "Score", "Grade"))
print("-" * 70) # Separator line

for entry in entries:
    name = entry["name"]
    domain = entry["domain"]
    score = entry["score"]
    grade = entry["grade"]
    print("{:<25} {:<25} {:<10} {:<5}".format(name, domain, score, grade))

Name                      Domain                    Score      Grade
----------------------------------------------------------------------
Amazon AWS                amazonaws.com             77         C    
Google Services           gmail.com                 54         F    
Datadog                   datadoghq.com             91         A    
Microsoft Azure           azure.com                 53         F    
Oracle Cloud              oraclecloud.com           79         C    


In [2]:
# get 3rd party vendors for each domain

def get_all_third_party_vendors(domain, headers=None, limit=500):
    """
    Retrieves all third-party vendors for a domain, handling pagination.

    Args:
        domain: The domain to query.
        headers: Optional headers for the request.
        limit: The number of results per page.

    Returns:
        A list containing the data from all pages.
    """

    all_entries = []
    page = 1

    while True:
        url = f"https://api.securityscorecard.io/vendor-detection/{domain}/third-party?page={page}&limit={limit}"
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()  # Check for HTTP errors
            data = response.json()

            if data["size"] == 0:
                break  # Stop if no entries are returned

            all_entries.extend(data["entries"])
            page += 1  # Increment the page number

        except requests.exceptions.RequestException as e:
            print(f"Error fetching page {page}: {e}")
            break #stop on error
        except json.JSONDecodeError as e:
            print(f"Error decoding json on page {page}: {e}")
            break #stop on error decoding json.

    return all_entries

for entry in entries:
    domain = entry["domain"]
    third_party_vendors = get_all_third_party_vendors(domain, headers=headers)

    if third_party_vendors:
        print(f"Found {len(third_party_vendors)} third-party vendors for {domain}.")
        # Process the third_party_vendors list here (e.g., print, save to file)
        # Example print of first 3 results.
        if len(third_party_vendors) > 0:
          print("Example first 3 results:")
          for i in range(min(3, len(third_party_vendors))):
            print(third_party_vendors[i])
    else:
        print(f"No third-party vendors found for {domain} or error occurred.")

Found 43 third-party vendors for amazonaws.com.
Example first 3 results:
{'domain': 'adobe.com', 'industry': 'technology', 'company': 'Adobe', 'source': 'Enhanced illumination', 'connection_detail': 'Job Posting', 'observed': '2025-04-10T13:37:05.000Z', 'score': 75}
{'domain': 'adobe.io', 'industry': 'information_services', 'company': 'Adobe', 'source': 'HTTP requests', 'connection_detail': 'https://pay.amazon.com/ calls POST - 846-rqb-314.mktoresp.com/webevents/visitWebPage?_mchNc=1745464192128&_mchCn=&_mchId=846-RQB-314&_mchTk=_mch-amazon.com-1745464192127-56412&_mchHo=pay.amazon.com&_mchPo=&_mchRu=%2F&_mchPc=https%3A&_mchVr=160&_mchEcid=&_mchHa=&_mchRe=&_mchQp= OK/200', 'observed': '2025-04-24T03:09:52.000Z', 'score': 77}
{'domain': 'alloy-software.com', 'industry': 'information_services', 'company': 'Alloy-software', 'source': 'Enhanced illumination', 'connection_detail': 'Job Posting', 'observed': '2025-04-10T13:37:05.000Z', 'score': 45}
Found 138 third-party vendors for gmail.com

In [3]:
# see how many 3rd party vendors overlap in your portfolio

from collections import defaultdict

vendor_counts = defaultdict(lambda: {"count": 0, "domains": [], "vendor_domain": None})

overlap_threshold = 3

for entry in entries:
    domain = entry["domain"]
    third_party_vendors = get_all_third_party_vendors(domain, headers=headers)
    if third_party_vendors:
        for vendor in third_party_vendors:
            vendor_name = vendor["company"]
            vendor_domain = vendor["domain"]
            vendor_counts[vendor_name]["count"] += 1
            vendor_counts[vendor_name]["domains"].append(domain)
            vendor_counts[vendor_name]["vendor_domain"] = vendor_domain  # Store vendor domain
    else:
        print(f"No third-party vendors found for {domain} or error occurred.")

sorted_vendors_third_party = sorted(vendor_counts.items(), key=lambda item: item[1]["count"], reverse=True)

print("{:<8} {:<30} {:<30} {}".format("Count", "Company", "Vendor Domain", "List of Companies with Vendor"))
print("-" * 120)

for vendor_name, data in sorted_vendors_third_party:
    if data['count'] > overlap_threshold:  # Check if count is greater than threshold number
        domains_str = ", ".join(data['domains'])
        vendor_domain_str = str(data['vendor_domain']) if data['vendor_domain'] is not None else "N/A"
        vendor_name_str = str(vendor_name) if vendor_name is not None else "N/A"
        print("{:<8} {:<30} {:<30} {}".format(data['count'], vendor_name_str, vendor_domain_str, domains_str))

Count    Company                        Vendor Domain                  List of Companies with Vendor
------------------------------------------------------------------------------------------------------------------------
7        Adobe                          adobe.com                      amazonaws.com, amazonaws.com, gmail.com, gmail.com, datadoghq.com, datadoghq.com, oraclecloud.com
5        Amazon                         amazon.com                     amazonaws.com, gmail.com, datadoghq.com, azure.com, oraclecloud.com
5        Google                         google.com                     amazonaws.com, gmail.com, datadoghq.com, azure.com, oraclecloud.com
5        Microsoft Corporation          microsoft.com                  amazonaws.com, gmail.com, datadoghq.com, azure.com, oraclecloud.com
5        Oracle America, Inc.           oracle.com                     amazonaws.com, gmail.com, datadoghq.com, azure.com, oraclecloud.com
4        DigitalOcean                   digitalocean.

In [4]:
# get 4th party vendors

def get_all_fourth_party_vendors(domain, headers=None, limit=500):
    """
    Retrieves all fourth-party vendors for a domain, handling pagination.

    Args:
        domain: The domain to query.
        headers: Optional headers for the request.
        limit: The number of results per page.

    Returns:
        A list containing the data from all pages.
    """

    all_entries = []
    page = 1

    while True:
        url = f"https://api.securityscorecard.io/vendor-detection/{domain}/fourth-party?page={page}&limit={limit}"
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()  # Check for HTTP errors
            data = response.json()

            if data["size"] == 0:
                break  # Stop if no entries are returned

            all_entries.extend(data["entries"])
            page += 1  # Increment the page number

        except requests.exceptions.RequestException as e:
            print(f"Error fetching page {page}: {e}")
            break  # Stop on error
        except json.JSONDecodeError as e:
            print(f"Error decoding json on page {page}: {e}")
            break  # Stop on error decoding json.

    return all_entries

#Example usage (assuming 'entries' and 'headers' are defined elsewhere):
for entry in entries:
    domain = entry["domain"]
    fourth_party_vendors = get_all_fourth_party_vendors(domain, headers=headers)

    if fourth_party_vendors:
        print(f"Found {len(fourth_party_vendors)} fourth-party vendors for {domain}.")
        # Process the fourth_party_vendors list here (e.g., print, save to file)
        # Example print of first 3 results.
        if len(fourth_party_vendors) > 0:
            print("Example first 3 results:")
            for i in range(min(3, len(fourth_party_vendors))):
                print(fourth_party_vendors[i])
    else:
        print(f"No fourth-party vendors found for {domain} or error occurred.")

Found 3469 fourth-party vendors for amazonaws.com.
Example first 3 results:
{'domain': '0.com', 'industry': 'technology', 'company': '0', 'source': 'Mail exchange', 'connection_detail': 'www.mobileway.com - 0 . - MX', 'observed': '2025-04-20T06:26:13.000Z', 'score': 100}
{'domain': '10xgenomics.com', 'industry': 'information_services', 'company': '10x Genomics, Inc.', 'source': 'Enhanced illumination', 'connection_detail': 'Job Posting', 'observed': '2025-04-10T13:37:05.000Z', 'score': 85}
{'domain': '1and1.com', 'industry': 'technology', 'company': '1and1', 'source': 'Mail exchange', 'connection_detail': 'ww1.myrampup.com - 10 mxint02.1and1.com. - MX', 'observed': '2025-04-12T18:11:57.000Z', 'score': 61}
Found 3728 fourth-party vendors for gmail.com.
Example first 3 results:
{'domain': '0.com', 'industry': 'technology', 'company': '0', 'source': 'Mail exchange', 'connection_detail': 'www.dronie.com - 0 . - MX', 'observed': '2025-04-29T13:25:42.000Z', 'score': 100}
{'domain': '10xgenom

In [5]:
# see how many overlap between 4th party

vendor_counts_fourth_party = defaultdict(lambda: {"count": 0, "domains":[], "vendor_domain": None})

for entry in entries:
    domain = entry["domain"]
    fourth_party_vendors = get_all_fourth_party_vendors(domain, headers=headers)
    if fourth_party_vendors:
        for vendor in fourth_party_vendors:
            vendor_name = vendor["company"]
            vendor_domain = vendor.get("domain")
            vendor_counts_fourth_party[vendor_name]["count"] += 1
            vendor_counts_fourth_party[vendor_name]["domains"].append(domain)
            vendor_counts_fourth_party[vendor_name]["vendor_domain"] = vendor_domain  # Store vendor domain
    else:
        print(f"No fourth-party vendors found for {domain} or error occurred.")

sorted_vendors_fourth_party = sorted(vendor_counts_fourth_party.items(), key=lambda item: item[1]["count"], reverse=True)

print("\n--- Fourth-Party Vendor Overlap ---")
print("{:<8} {:<30} {:<30} {}".format("Count", "Company", "Vendor Domain", "List of Companies with 4th Party Vendor"))
print("-" * 120)

line_print_count = 0

for vendor_name, data in sorted_vendors_fourth_party:
    if data['count'] > 5:  # Check if count is greater than threshold number
        domains_str = ", ".join(data['domains'])
        vendor_domain_str = str(data['vendor_domain']) if data['vendor_domain'] is not None else "N/A"
        vendor_name_str = str(vendor_name) if vendor_name is not None else "N/A"
        print("{:<8} {:<30} {:<30} {}".format(data['count'], vendor_name_str, vendor_domain_str, domains_str))

        line_print_count += 1
        # optionally limit result printout size
        if line_print_count >= 20:
            break;


--- Fourth-Party Vendor Overlap ---
Count    Company                        Vendor Domain                  List of Companies with 4th Party Vendor
------------------------------------------------------------------------------------------------------------------------
30       N/A                            teamtito.com                   amazonaws.com, amazonaws.com, amazonaws.com, amazonaws.com, amazonaws.com, amazonaws.com, gmail.com, gmail.com, gmail.com, gmail.com, gmail.com, gmail.com, gmail.com, gmail.com, datadoghq.com, datadoghq.com, datadoghq.com, datadoghq.com, datadoghq.com, datadoghq.com, datadoghq.com, azure.com, azure.com, azure.com, azure.com, oraclecloud.com, oraclecloud.com, oraclecloud.com, oraclecloud.com, oraclecloud.com
13       Yandex                         yandex.ru                      amazonaws.com, amazonaws.com, amazonaws.com, gmail.com, gmail.com, gmail.com, datadoghq.com, datadoghq.com, datadoghq.com, azure.com, oraclecloud.com, oraclecloud.com, oracleclou

In [ ]:
# output all third party and fourth party dependencies

vendor_objects = []

# Populate vendor_objects with third-party data
for vendor_name, data in sorted_vendors_third_party:
    vendor_objects.append({
        "company": vendor_name,
        "domain": data['vendor_domain'],
        "third_party_count": data['count'],
        "fourth_party_count": 0  # Initialize fourth-party count to 0
    })

# Update fourth-party counts from sorted_vendors_fourth_party
for vendor_name, data in sorted_vendors_fourth_party:
    found = False
    for vendor in vendor_objects:
        if vendor['company'] == vendor_name:
            vendor['fourth_party_count'] = data['count']
            found = True
            break  # Vendor found, no need to continue inner loop
    if not found: # if vendor not found, add it.
        vendor_objects.append({
            "company": vendor_name,
            "domain": data['vendor_domain'],
            "third_party_count": 0,
            "fourth_party_count": data['count']
        })

line_print_count = 0;

# Print the new list of objects (optional)
for vendor in vendor_objects:
    if vendor.get('third_party_count', 0) > overlap_threshold or vendor.get('fourth_party_count', 0) > overlap_threshold + 2:
        print(f"Company: {vendor['company']}, Domain: {vendor['domain']}, Third Party Dependencies: {vendor['third_party_count']}, Fourth Party Dependencies: {vendor['fourth_party_count']}")

        line_print_count += 1
        # optionally limit result printout size
        if line_print_count >= 20:
            break;

Company: Adobe, Domain: adobe.com, Third Party Dependencies: 7, Fourth Party Dependencies: 10
Company: Amazon, Domain: amazon.com, Third Party Dependencies: 5, Fourth Party Dependencies: 10
Company: Google, Domain: google.com, Third Party Dependencies: 5, Fourth Party Dependencies: 5
Company: Microsoft Corporation, Domain: microsoft.com, Third Party Dependencies: 5, Fourth Party Dependencies: 5
Company: Oracle America, Inc., Domain: oracle.com, Third Party Dependencies: 5, Fourth Party Dependencies: 5
Company: DigitalOcean, Domain: digitalocean.com, Third Party Dependencies: 4, Fourth Party Dependencies: 5
Company: Facebook, Domain: facebook.com, Third Party Dependencies: 4, Fourth Party Dependencies: 5
Company: Python Software Foundation Corp, Domain: python.org, Third Party Dependencies: 4, Fourth Party Dependencies: 5
Company: Cloudflare, Inc., Domain: cloudflare.com, Third Party Dependencies: 4, Fourth Party Dependencies: 5
Company: GitHub, Domain: github.com, Third Party Dependenc